# League Insights & Fun Stats

Answering your burning questions about League of Inches history.

In [ ]:
spark.sql("USE CATALOG workspace")

## 1. Worst Draft Pick Ever

Which draft pick scored the fewest career points?

In [ ]:
worst_pick = spark.sql("""
WITH draft_picks AS (
  SELECT
    dr.season,
    CAST(dp.round AS INT) AS round,
    dp.pick_no,
    dp.roster_id,
    dp.player_id,
    dp.draft_id,
    lower(regexp_replace(coalesce(li.name, 'unknown'), '[^a-zA-Z0-9]+', '_')) AS cluster_key
  FROM workspace.sleeper_raw.sleeper_draft_picks_snapshot dp
  JOIN workspace.sleeper_raw.sleeper_drafts_snapshot dr ON dp.draft_id = dr.draft_id
  JOIN workspace.sleeper_raw.sleeper_league_info_snapshot li ON dr.league_id = li.league_id
  WHERE dp.player_id IS NOT NULL
),
career_points AS (
  SELECT
    p.cluster_key,
    p.season AS draft_season,
    p.round,
    p.pick_no,
    p.roster_id,
    p.player_id,
    pl.full_name,
    pl.position,
    COALESCE(SUM(pw.points), 0) AS total_career_points,
    COUNT(DISTINCT pw.season) AS seasons_played,
    COUNT(*) AS games_played
  FROM draft_picks p
  LEFT JOIN workspace.sleeper_core.dim_players pl ON p.player_id = pl.player_id
  LEFT JOIN workspace.sleeper_core.fact_player_week_enriched pw
    ON p.cluster_key = pw.cluster_key
    AND p.player_id = pw.player_id
    AND CAST(pw.season AS INT) >= CAST(p.season AS INT)
  GROUP BY p.cluster_key, p.season, p.round, p.pick_no, p.roster_id, p.player_id, pl.full_name, pl.position
)
SELECT
  cp.*,
  mr.manager_display_name AS drafted_by,
  CONCAT(draft_season, ' Round ', round, ' Pick ', pick_no) AS pick_description
FROM career_points cp
LEFT JOIN workspace.sleeper_core.dim_manager_roster_map mr
  ON cp.roster_id = mr.roster_id
  AND cp.cluster_key = mr.cluster_key
  AND cp.draft_season = mr.season
WHERE cp.cluster_key = 'league_of_inches'
ORDER BY total_career_points ASC, round ASC
LIMIT 20
""")

display(worst_pick)

## 2. Top Scorers Per Team

Which player scored the most **starting lineup points** for each team?

This only counts points when the player was actually started, not bench points.

In [ ]:
# All positions - FULL CAREER across all seasons (STARTED POINTS ONLY)
top_scorers_all = spark.sql("""
WITH player_games_started AS (
  -- Count actual games started per player per manager
  SELECT
    mr.manager_display_name,
    mr.cluster_key,
    pw.player_id,
    COUNT(*) AS games_started
  FROM workspace.sleeper_core.fact_player_week pw
  JOIN workspace.sleeper_core.fact_player_week_enriched pwe
    ON pw.league_id = pwe.league_id
    AND pw.season = pwe.season
    AND pw.week = pwe.week
    AND pw.player_id = pwe.player_id
  LEFT JOIN workspace.sleeper_core.dim_manager_roster_map mr
    ON pw.league_id = mr.league_id
    AND pw.roster_id = mr.roster_id
    AND pw.season = mr.season
  WHERE mr.cluster_key = 'league_of_inches'
    AND pw.was_started = TRUE
  GROUP BY mr.manager_display_name, mr.cluster_key, pw.player_id
),
player_career_by_manager AS (
  -- Aggregate across all league_ids (seasons) for each manager
  SELECT
    mr.manager_display_name,
    mr.cluster_key,
    pt.player_id,
    MAX(pt.full_name) AS player_name,
    MAX(pt.position) AS position,
    SUM(pt.points_as_starter) AS total_points,  -- Only started points
    SUM(pt.weeks_count) AS weeks_count,
    MIN(pt.first_season) AS first_season,
    MAX(pt.last_season) AS last_season
  FROM workspace.sleeper_core.agg_player_roster_totals pt
  LEFT JOIN workspace.sleeper_core.dim_manager_roster_map mr
    ON pt.league_id = mr.league_id
    AND pt.roster_id = mr.roster_id
  WHERE mr.cluster_key = 'league_of_inches'
  GROUP BY mr.manager_display_name, mr.cluster_key, pt.player_id
),
with_games AS (
  SELECT
    pc.*,
    COALESCE(gs.games_started, 0) AS games_started
  FROM player_career_by_manager pc
  LEFT JOIN player_games_started gs
    ON pc.manager_display_name = gs.manager_display_name
    AND pc.cluster_key = gs.cluster_key
    AND pc.player_id = gs.player_id
),
ranked AS (
  SELECT
    *,
    ROW_NUMBER() OVER (PARTITION BY manager_display_name ORDER BY total_points DESC) AS rank
  FROM with_games
)
SELECT
  manager_display_name,
  player_name,
  position,
  ROUND(total_points, 1) AS points_when_started,
  games_started,
  ROUND(total_points / NULLIF(games_started, 0), 2) AS ppg_when_started,
  weeks_count AS weeks_on_roster,
  first_season,
  last_season
FROM ranked
WHERE rank = 1
ORDER BY points_when_started DESC
""")

display(top_scorers_all)

In [ ]:
# Excluding QBs - FULL CAREER across all seasons (STARTED POINTS ONLY)
top_scorers_no_qb = spark.sql("""
WITH player_games_started AS (
  -- Count actual games started per player per manager
  SELECT
    mr.manager_display_name,
    mr.cluster_key,
    pw.player_id,
    COUNT(*) AS games_started
  FROM workspace.sleeper_core.fact_player_week pw
  JOIN workspace.sleeper_core.fact_player_week_enriched pwe
    ON pw.league_id = pwe.league_id
    AND pw.season = pwe.season
    AND pw.week = pwe.week
    AND pw.player_id = pwe.player_id
  LEFT JOIN workspace.sleeper_core.dim_manager_roster_map mr
    ON pw.league_id = mr.league_id
    AND pw.roster_id = mr.roster_id
    AND pw.season = mr.season
  LEFT JOIN workspace.sleeper_core.dim_players p ON pw.player_id = p.player_id
  WHERE mr.cluster_key = 'league_of_inches'
    AND pw.was_started = TRUE
    AND p.position != 'QB'
  GROUP BY mr.manager_display_name, mr.cluster_key, pw.player_id
),
player_career_by_manager AS (
  -- Aggregate across all league_ids (seasons) for each manager
  SELECT
    mr.manager_display_name,
    mr.cluster_key,
    pt.player_id,
    MAX(pt.full_name) AS player_name,
    MAX(pt.position) AS position,
    SUM(pt.points_as_starter) AS total_points,  -- Only started points
    SUM(pt.weeks_count) AS weeks_count,
    MIN(pt.first_season) AS first_season,
    MAX(pt.last_season) AS last_season
  FROM workspace.sleeper_core.agg_player_roster_totals pt
  LEFT JOIN workspace.sleeper_core.dim_manager_roster_map mr
    ON pt.league_id = mr.league_id
    AND pt.roster_id = mr.roster_id
  WHERE mr.cluster_key = 'league_of_inches'
    AND pt.position != 'QB'
  GROUP BY mr.manager_display_name, mr.cluster_key, pt.player_id
),
with_games AS (
  SELECT
    pc.*,
    COALESCE(gs.games_started, 0) AS games_started
  FROM player_career_by_manager pc
  LEFT JOIN player_games_started gs
    ON pc.manager_display_name = gs.manager_display_name
    AND pc.cluster_key = gs.cluster_key
    AND pc.player_id = gs.player_id
),
ranked AS (
  SELECT
    *,
    ROW_NUMBER() OVER (PARTITION BY manager_display_name ORDER BY total_points DESC) AS rank
  FROM with_games
)
SELECT
  manager_display_name,
  player_name,
  position,
  ROUND(total_points, 1) AS points_when_started,
  games_started,
  ROUND(total_points / NULLIF(games_started, 0), 2) AS ppg_when_started,
  weeks_count AS weeks_on_roster,
  first_season,
  last_season
FROM ranked
WHERE rank = 1
ORDER BY points_when_started DESC
""")

display(top_scorers_no_qb)

## 3. Start/Sit Performance Splits

Did you leave points on the bench? This analysis compares how players performed when you **started** them vs when they sat on your **bench**.

**Positive bench_advantage** = Player scored more PPG on your bench (you should have started them!)  
**Negative bench_advantage** = Player scored more PPG when started (good job starting them!)

Shows players with at least 5 games started AND 5 games benched across all League of Inches history.

In [ ]:
start_sit_splits = spark.sql("""
WITH player_splits AS (
  SELECT
    pw.player_id,
    p.full_name,
    p.position,
    -- Started stats
    COUNT(CASE WHEN pw.was_started THEN 1 END) AS games_started,
    COALESCE(AVG(CASE WHEN pw.was_started THEN pw.points END), 0) AS ppg_started,
    COALESCE(SUM(CASE WHEN pw.was_started THEN pw.points ELSE 0 END), 0) AS total_started,
    -- Benched stats
    COUNT(CASE WHEN NOT pw.was_started THEN 1 END) AS games_benched,
    COALESCE(AVG(CASE WHEN NOT pw.was_started THEN pw.points END), 0) AS ppg_benched,
    COALESCE(SUM(CASE WHEN NOT pw.was_started THEN pw.points ELSE 0 END), 0) AS total_benched
  FROM workspace.sleeper_core.fact_player_week pw
  LEFT JOIN workspace.sleeper_core.dim_players p ON pw.player_id = p.player_id
  JOIN workspace.sleeper_core.fact_player_week_enriched pwe 
    ON pw.league_id = pwe.league_id
    AND pw.season = pwe.season
    AND pw.week = pwe.week
    AND pw.player_id = pwe.player_id
  WHERE pwe.cluster_key = 'league_of_inches'
  GROUP BY pw.player_id, p.full_name, p.position
  HAVING COUNT(*) >= 10  -- At least 10 games
)
SELECT
  full_name,
  position,
  games_started,
  ROUND(ppg_started, 2) AS ppg_started,
  games_benched,
  ROUND(ppg_benched, 2) AS ppg_benched,
  ROUND(ppg_benched - ppg_started, 2) AS bench_advantage,
  games_started + games_benched AS total_games
FROM player_splits
WHERE games_benched >= 5 AND games_started >= 5  -- Need both to compare
ORDER BY bench_advantage DESC
LIMIT 30
""")

display(start_sit_splits)

### Worst When Started (Should Have Benched Them!)

In [ ]:
worst_when_started = spark.sql("""
WITH player_splits AS (
  SELECT
    pw.player_id,
    p.full_name,
    p.position,
    COUNT(CASE WHEN pw.was_started THEN 1 END) AS games_started,
    COALESCE(AVG(CASE WHEN pw.was_started THEN pw.points END), 0) AS ppg_started,
    COUNT(CASE WHEN NOT pw.was_started THEN 1 END) AS games_benched,
    COALESCE(AVG(CASE WHEN NOT pw.was_started THEN pw.points END), 0) AS ppg_benched
  FROM workspace.sleeper_core.fact_player_week pw
  LEFT JOIN workspace.sleeper_core.dim_players p ON pw.player_id = p.player_id
  JOIN workspace.sleeper_core.fact_player_week_enriched pwe 
    ON pw.league_id = pwe.league_id
    AND pw.season = pwe.season
    AND pw.week = pwe.week
    AND pw.player_id = pwe.player_id
  WHERE pwe.cluster_key = 'league_of_inches'
  GROUP BY pw.player_id, p.full_name, p.position
  HAVING COUNT(*) >= 10
)
SELECT
  full_name,
  position,
  games_started,
  ROUND(ppg_started, 2) AS ppg_started,
  games_benched,
  ROUND(ppg_benched, 2) AS ppg_benched,
  ROUND(ppg_benched - ppg_started, 2) AS bench_advantage
FROM player_splits
WHERE games_benched >= 5 AND games_started >= 5
ORDER BY ppg_started ASC, bench_advantage DESC
LIMIT 20
""")

display(worst_when_started)

## 4. Average Team "Age" (Roster Tenure)

How long have players been on each roster on average?

In [ ]:
roster_age = spark.sql("""
-- Calculate average roster tenure: how long has each player been with their current manager?
WITH current_season AS (
  SELECT MAX(season) AS max_season 
  FROM workspace.sleeper_raw.sleeper_league_info_snapshot
),
current_rosters AS (
  -- Get each manager's current roster (most recent season)
  SELECT DISTINCT
    mr.manager_display_name,
    mr.cluster_key,
    explode(r.players) AS player_id
  FROM workspace.sleeper_raw.sleeper_rosters_snapshot r
  JOIN workspace.sleeper_raw.sleeper_league_info_snapshot li ON r.league_id = li.league_id
  JOIN workspace.sleeper_core.dim_manager_roster_map mr 
    ON r.league_id = mr.league_id 
    AND r.roster_id = mr.roster_id
    AND li.season = mr.season
  CROSS JOIN current_season cs
  WHERE lower(regexp_replace(coalesce(li.name, 'unknown'), '[^a-zA-Z0-9]+', '_')) = 'league_of_inches'
    AND li.season = cs.max_season
),
manager_ownership_history AS (
  -- Get ALL ownership periods for each manager/player combo across all seasons
  SELECT DISTINCT
    mr.manager_display_name,
    mr.cluster_key,
    po.player_id,
    po.ownership_id,
    po.season,
    po.acquired_via,
    po.departed_via
  FROM workspace.sleeper_core.dim_player_ownership po
  JOIN workspace.sleeper_core.dim_manager_roster_map mr
    ON po.league_id = mr.league_id
    AND po.roster_id = mr.roster_id
    AND po.season = mr.season
  WHERE mr.cluster_key = 'league_of_inches'
),
player_tenure AS (
  -- For each player on current roster, find when they were FIRST acquired by this manager
  SELECT
    cr.manager_display_name,
    cr.player_id,
    p.full_name,
    p.position,
    MIN(moh.season) AS first_acquired_season,  -- First season manager owned this player
    COUNT(DISTINCT moh.ownership_id) AS times_acquired,  -- Separate ownership periods
    -- Only count as boomerang if they were actually DROPPED (departed_via is not null)
    COUNT(DISTINCT CASE WHEN moh.departed_via IS NOT NULL THEN moh.ownership_id END) AS times_dropped
  FROM current_rosters cr
  LEFT JOIN manager_ownership_history moh
    ON cr.player_id = moh.player_id
    AND cr.manager_display_name = moh.manager_display_name
    AND cr.cluster_key = moh.cluster_key
  LEFT JOIN workspace.sleeper_core.dim_players p ON cr.player_id = p.player_id
  GROUP BY cr.manager_display_name, cr.player_id, p.full_name, p.position
),
roster_summary AS (
  -- Aggregate tenure stats per manager
  SELECT
    pt.manager_display_name,
    COUNT(DISTINCT pt.player_id) AS total_players,
    -- Average years each player has been with the team
    AVG(CAST((SELECT max_season FROM current_season) AS INT) - CAST(pt.first_acquired_season AS INT)) AS avg_seasons_on_roster,
    -- Count "boomerang" players: acquired multiple times AND dropped at least once
    COUNT(CASE WHEN times_acquired > 1 AND times_dropped > 0 THEN 1 END) AS players_reacquired
  FROM player_tenure pt
  GROUP BY pt.manager_display_name
)
SELECT
  rs.manager_display_name,
  rs.total_players,
  ROUND(rs.avg_seasons_on_roster, 2) AS avg_seasons_tenure,
  rs.players_reacquired AS boomerang_players,  -- Dropped then re-acquired
  ROUND(rs.players_reacquired * 100.0 / rs.total_players, 1) AS pct_boomerang
FROM roster_summary rs
ORDER BY avg_seasons_tenure DESC
""")

display(roster_age)

In [ ]:
boomerang_details = spark.sql("""
-- Show which players each manager dropped and re-acquired
WITH current_season AS (
  SELECT MAX(season) AS max_season 
  FROM workspace.sleeper_raw.sleeper_league_info_snapshot
),
current_rosters AS (
  -- Get each manager's current roster (most recent season)
  SELECT DISTINCT
    mr.manager_display_name,
    mr.cluster_key,
    explode(r.players) AS player_id
  FROM workspace.sleeper_raw.sleeper_rosters_snapshot r
  JOIN workspace.sleeper_raw.sleeper_league_info_snapshot li ON r.league_id = li.league_id
  JOIN workspace.sleeper_core.dim_manager_roster_map mr 
    ON r.league_id = mr.league_id 
    AND r.roster_id = mr.roster_id
    AND li.season = mr.season
  CROSS JOIN current_season cs
  WHERE lower(regexp_replace(coalesce(li.name, 'unknown'), '[^a-zA-Z0-9]+', '_')) = 'league_of_inches'
    AND li.season = cs.max_season
),
manager_ownership_history AS (
  -- Get ALL ownership periods for each manager/player combo across all seasons
  SELECT DISTINCT
    mr.manager_display_name,
    mr.cluster_key,
    po.player_id,
    po.ownership_id,
    po.season,
    po.acquired_via,
    po.departed_via
  FROM workspace.sleeper_core.dim_player_ownership po
  JOIN workspace.sleeper_core.dim_manager_roster_map mr
    ON po.league_id = mr.league_id
    AND po.roster_id = mr.roster_id
    AND po.season = mr.season
  WHERE mr.cluster_key = 'league_of_inches'
),
player_boomerangs AS (
  SELECT
    cr.manager_display_name,
    cr.player_id,
    p.full_name,
    p.position,
    MIN(moh.season) AS first_acquired_season,
    MAX(moh.season) AS last_acquired_season,
    COUNT(DISTINCT moh.ownership_id) AS times_acquired,
    COUNT(DISTINCT CASE WHEN moh.departed_via IS NOT NULL THEN moh.ownership_id END) AS times_dropped,
    -- Get all acquisition methods
    COLLECT_SET(moh.acquired_via) AS acquisition_methods,
    -- Get all departure methods
    COLLECT_SET(moh.departed_via) AS departure_methods
  FROM current_rosters cr
  LEFT JOIN manager_ownership_history moh
    ON cr.player_id = moh.player_id
    AND cr.manager_display_name = moh.manager_display_name
    AND cr.cluster_key = moh.cluster_key
  LEFT JOIN workspace.sleeper_core.dim_players p ON cr.player_id = p.player_id
  GROUP BY cr.manager_display_name, cr.player_id, p.full_name, p.position
  HAVING COUNT(DISTINCT moh.ownership_id) > 1 
    AND COUNT(DISTINCT CASE WHEN moh.departed_via IS NOT NULL THEN moh.ownership_id END) > 0
)
SELECT
  manager_display_name,
  full_name,
  position,
  times_acquired,
  times_dropped,
  first_acquired_season,
  last_acquired_season,
  CONCAT_WS(', ', acquisition_methods) AS how_acquired,
  CONCAT_WS(', ', departure_methods) AS how_departed
FROM player_boomerangs
ORDER BY manager_display_name, times_acquired DESC, full_name
""")

display(boomerang_details)

### Boomerang Players Breakdown

See which specific players each manager dropped and re-acquired.

## 5. Most Rostered Player

Which player has been on the most different teams?

In [ ]:
most_rostered = spark.sql("""
WITH player_rosters AS (
  SELECT
    po.player_id,
    p.full_name,
    p.position,
    COUNT(DISTINCT po.roster_id) AS unique_rosters,
    COUNT(DISTINCT po.ownership_id) AS total_acquisitions,
    MIN(po.season) AS first_season,
    MAX(po.season) AS last_season,
    -- Get list of managers who owned them
    COLLECT_SET(mr.manager_display_name) AS managers
  FROM workspace.sleeper_core.dim_player_ownership po
  LEFT JOIN workspace.sleeper_core.dim_players p ON po.player_id = p.player_id
  LEFT JOIN workspace.sleeper_core.dim_manager_roster_map mr
    ON po.league_id = mr.league_id
    AND po.season = mr.season
    AND po.roster_id = mr.roster_id
  JOIN workspace.sleeper_raw.sleeper_league_info_snapshot li ON po.league_id = li.league_id
  WHERE lower(regexp_replace(coalesce(li.name, 'unknown'), '[^a-zA-Z0-9]+', '_')) = 'league_of_inches'
  GROUP BY po.player_id, p.full_name, p.position
)
SELECT
  full_name,
  position,
  unique_rosters,
  total_acquisitions,
  first_season,
  last_season,
  ROUND(total_acquisitions * 1.0 / unique_rosters, 2) AS avg_acquisitions_per_roster,
  CONCAT_WS(', ', managers) AS owned_by
FROM player_rosters
WHERE unique_rosters > 1  -- Exclude players only ever on one roster
ORDER BY unique_rosters DESC, total_acquisitions DESC
LIMIT 30
""")

display(most_rostered)

## 6. Single-Team Loyalty (Churned but Faithful)

Players who have only ever been on ONE team, but were dropped and re-acquired multiple times.

In [ ]:
single_team_churned = spark.sql("""
WITH player_rosters AS (
  SELECT
    po.player_id,
    p.full_name,
    p.position,
    COUNT(DISTINCT po.roster_id) AS unique_rosters,
    COUNT(DISTINCT po.ownership_id) AS total_acquisitions,
    MIN(po.season) AS first_season,
    MAX(po.season) AS last_season,
    ANY_VALUE(po.roster_id) AS roster_id,
    -- Count how many times departed and came back
    COUNT(CASE WHEN po.departed_via IS NOT NULL THEN 1 END) AS times_departed
  FROM workspace.sleeper_core.dim_player_ownership po
  LEFT JOIN workspace.sleeper_core.dim_players p ON po.player_id = p.player_id
  JOIN workspace.sleeper_raw.sleeper_league_info_snapshot li ON po.league_id = li.league_id
  WHERE lower(regexp_replace(coalesce(li.name, 'unknown'), '[^a-zA-Z0-9]+', '_')) = 'league_of_inches'
  GROUP BY po.player_id, p.full_name, p.position
  HAVING COUNT(DISTINCT po.roster_id) = 1  -- Only ever on ONE roster
    AND COUNT(DISTINCT po.ownership_id) > 1  -- But acquired multiple times
)
SELECT
  pr.full_name,
  pr.position,
  mr.manager_display_name AS loyal_to,
  pr.total_acquisitions,
  pr.times_departed,
  pr.first_season,
  pr.last_season,
  CASE 
    WHEN pr.times_departed = 0 THEN 'Never left'
    WHEN pr.times_departed = pr.total_acquisitions - 1 THEN 'All drops'
    ELSE CONCAT(pr.times_departed, ' departures')
  END AS churn_pattern
FROM player_rosters pr
LEFT JOIN workspace.sleeper_core.dim_manager_roster_map mr
  ON pr.roster_id = mr.roster_id
WHERE mr.cluster_key = 'league_of_inches'
  AND mr.season = (SELECT MAX(season) FROM workspace.sleeper_raw.sleeper_league_info_snapshot)
ORDER BY pr.total_acquisitions DESC, pr.times_departed DESC
LIMIT 30
""")

display(single_team_churned)

---

## Summary

This notebook answers:
1. ✅ Worst draft picks (least career points)
2. ✅ Top scorers per team (with/without QB)
3. ✅ Start/sit performance splits (bench vs started PPG)
4. ✅ Average roster tenure (how long players stay)
5. ✅ Most rostered player (been on most teams)
6. ✅ Single-team loyalty (churned but always came back)

All queries use the `league_of_inches` cluster key for filtering.